E-commerce Text Classification with RNNs and Multiple Embedding Techniques

In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.7 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam


In [4]:
import csv

# Use quoting=csv.QUOTE_NONE to handle unclosed quotes (like inch marks) in descriptions
# The dataset typically doesn't have a header, so we provide names
train_df = pd.read_csv(
    "/content/drive/MyDrive/data/ecommerceDataset.csv",
    names=["label", "text"],
    header=None,
    quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

print(f"Loaded {len(train_df)} rows.")
train_df.head()

Loaded 35664 rows.


label  \
Household "Paper Plane Design Framed Wall Hanging Motivat... you would be proud to possess this unique paint... with only the and inks and precision epson          roland and hp printers. This innovative hd pri...  to achieve brilliant and true colours. Due to ...   our prints retain their beautiful colours for...   
          "SAF 'Floral' Framed Painting (Wood                30 inch x 10 inch                                  Special Effect UV Print Textured                    SAO297) Painting made up in synthetic frame wi... NaN                                                                                               NaN   
          "SAF 'UV Textured Modern Art Print Framed' Pain... 35 cm x 50 cm x 3 cm                               Set of 3) Color:Multicolor                     ...  the end product will be a picture that can spe...  it does not include glass along with the frame...                and shopping is just a click away!"   
          "Incredible Gifts India Wooden Happy Birthday U... Which Is Quite Solid With Light Particle Patter... Some Can Be Used As Table Top And The Big Sizes... NaN                                                NaN                                                                                               NaN   
          "Paper Plane Design Starry Night Vangoh Wall Ar... with only the best and original inks and precis... to achieve brilliant and true colours. Due to t...  our Canvas prints retain their beautiful colou...  to ensure that the colours of your original im...   with brilliant tones. Add colour and style to...   

                                                                                                                                                                                                                                                                                                                       text  
Household "Paper Plane Design Framed Wall Hanging Motivat... you would be proud to possess this unique paint... with only the and inks and precision epson          roland and hp printers. This innovative hd pri...  to achieve brilliant and true colours. Due to ...                               office or any room."  
          "SAF 'Floral' Framed Painting (Wood                30 inch x 10 inch                                  Special Effect UV Print Textured                    SAO297) Painting made up in synthetic frame wi... NaN                                                                                               NaN  
          "SAF 'UV Textured Modern Art Print Framed' Pain... 35 cm x 50 cm x 3 cm                               Set of 3) Color:Multicolor                     ...  the end product will be a picture that can spe...  it does not include glass along with the frame...                                                NaN  
          "Incredible Gifts India Wooden Happy Birthday U... Which Is Quite Solid With Light Particle Patter... Some Can Be Used As Table Top And The Big Sizes... NaN                                                NaN                                                                                               NaN  
          "Paper Plane Design Starry Night Vangoh Wall Ar... with only the best and original inks and precis... to achieve brilliant and true colours. Due to t...  our Canvas prints retain their beautiful colou...  to ensure that the colours of your original im...   office or any room. A perfect size of 36 inch...

In [5]:
print(train_df.shape)
print(train_df['label'].value_counts())


(35664, 2)
label
India"                                                                                                                                                                                                                                                                                                                                      22
khakis and hoodies. Our versatile and stylish casual clothing will enable you to fully participate in an active and comfortable lifestyle."                                                                                                                                                                                                 19
no shrink fabric."                                                                                                                                                                                                                                                                                                       

In [6]:
def clean_text(text):
    # Check if text is a string to avoid AttributeErrors with NaN/float values
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text

# Drop rows with missing values and use .copy() to avoid SettingWithCopyWarning
train_df = train_df.dropna(subset=["text"]).copy()
train_df["clean_text"] = train_df["text"].apply(clean_text)

In [7]:
le = LabelEncoder()
y = le.fit_transform(train_df["label"])
y = to_categorical(y)

num_classes = y.shape[1]


EXPERIMENT 1: RNN + TF-IDF

In [8]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(train_df["clean_text"]).toarray()


In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


In [10]:
X_train_rnn = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_val_rnn   = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))


In [11]:
tfidf_param_grid = [
    {"rnn_units": 64,  "dropout": 0.3, "lr": 1e-3},
    {"rnn_units": 128, "dropout": 0.3, "lr": 1e-3},
    {"rnn_units": 128, "dropout": 0.5, "lr": 1e-3},
    {"rnn_units": 128, "dropout": 0.5, "lr": 5e-4},
]


In [13]:
tfidf_results = []

for params in tfidf_param_grid:
    model = Sequential([
        SimpleRNN(
            params["rnn_units"],
            input_shape=(1, X_train.shape[1])
        ),
        Dropout(params["dropout"]),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(learning_rate=params["lr"]),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train_rnn, y_train,
        validation_data=(X_val_rnn, y_val),
        epochs=30,
        batch_size=64,
        verbose=0
    )

    tfidf_results.append({
        **params,
        "best_val_accuracy": max(history.history["val_accuracy"])
    })


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
tfidf_results_df = pd.DataFrame(tfidf_results)
tfidf_results_df.sort_values("best_val_accuracy", ascending=False)

,rnn_units,dropout,lr,best_val_accuracy
1,128,0.3,0.0010,0.681515
2,128,0.5,0.0010,0.674833
0,64,0.3,0.0010,0.648107
3,128,0.5,0.0005,0.543430


EXPERIMENT 2: RNN + Skip-gram (Word2Vec)

In [15]:
sentences = [text.split() for text in train_df["clean_text"]]


In [16]:
w2v_skipgram = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,   # Skip-gram
    epochs=50
)


In [17]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df["clean_text"])

sequences = tokenizer.texts_to_sequences(train_df["clean_text"])
max_len = 50
X_seq = pad_sequences(sequences, maxlen=max_len)


In [18]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in tokenizer.word_index.items():
    if word in w2v_skipgram.wv:
        embedding_matrix[idx] = w2v_skipgram.wv[word]


In [19]:
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y, test_size=0.2, random_state=42
)


In [20]:
skipgram_param_grid = [
    {"lstm_units": 64,  "dropout": 0.3, "lr": 1e-3},
    {"lstm_units": 128, "dropout": 0.3, "lr": 1e-3},
    {"lstm_units": 128, "dropout": 0.5, "lr": 1e-3},
    {"lstm_units": 128, "dropout": 0.5, "lr": 5e-4},
]


In [21]:
skipgram_results = []

for params in skipgram_param_grid:
    model = Sequential([
        Embedding(
            vocab_size,
            embedding_dim,
            weights=[embedding_matrix],
            input_length=max_len,
            trainable=False
        ),
        LSTM(params["lstm_units"]),
        Dropout(params["dropout"]),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(params["lr"]),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=8,
        batch_size=64,
        verbose=0
    )

    skipgram_results.append({
        **params,
        "best_val_accuracy": max(history.history["val_accuracy"])
    })


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
skipgram_results_df = pd.DataFrame(skipgram_results)
skipgram_results_df.sort_values("best_val_accuracy", ascending=False)


,lstm_units,dropout,lr,best_val_accuracy
1,128,0.3,0.0010,0.354120
2,128,0.5,0.0010,0.334076
0,64,0.3,0.0010,0.253898
3,128,0.5,0.0005,0.200445


EXPERIMENT 3: RNN + CBOW (Word2Vec)

In [23]:
w2v_cbow = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=0,   # CBOW
    epochs=10
)


In [24]:
embedding_matrix_cbow = np.zeros((vocab_size, embedding_dim))

for word, idx in tokenizer.word_index.items():
    if word in w2v_cbow.wv:
        embedding_matrix_cbow[idx] = w2v_cbow.wv[word]


In [25]:
cbow_results = []

for params in skipgram_param_grid:
    model = Sequential([
        Embedding(
            vocab_size,
            embedding_dim,
            weights=[embedding_matrix_cbow],
            input_length=max_len,
            trainable=False
        ),
        LSTM(params["lstm_units"]),
        Dropout(params["dropout"]),
        Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=Adam(params["lr"]),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=64,
        verbose=0
    )

    cbow_results.append({
        **params,
        "best_val_accuracy": max(history.history["val_accuracy"])
    })


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
pd.DataFrame(cbow_results).sort_values("best_val_accuracy", ascending=False)

,lstm_units,dropout,lr,best_val_accuracy
1,128,0.3,0.0010,0.547884
2,128,0.5,0.0010,0.532294
0,64,0.3,0.0010,0.305122
3,128,0.5,0.0005,0.273942


Comparison table

In [31]:

# Extract the best results from each experiment list
best_tfidf = max(tfidf_results, key=lambda x: x['best_val_accuracy']).copy()
best_tfidf['Model'] = 'RNN + TF-IDF'

best_skipgram = max(skipgram_results, key=lambda x: x['best_val_accuracy']).copy()
best_skipgram['Model'] = 'LSTM + Skip-gram'

best_cbow = max(cbow_results, key=lambda x: x['best_val_accuracy']).copy()
best_cbow['Model'] = 'LSTM + CBOW'

# Combine into a summary DataFrame
comparison_df = pd.DataFrame([best_tfidf, best_skipgram, best_cbow])

# Reorder columns for readability
columns_order = ['Model', 'best_val_accuracy', 'rnn_units', 'lstm_units', 'dropout', 'lr']
existing_cols = [c for c in columns_order if c in comparison_df.columns]

comparison_df[existing_cols].sort_values('best_val_accuracy', ascending=False)

,Model,best_val_accuracy,rnn_units,lstm_units,dropout,lr
0,RNN + TF-IDF,0.681515,128.0,NaN,0.3,0.001
2,LSTM + CBOW,0.547884,NaN,128.0,0.3,0.001
1,LSTM + Skip-gram,0.354120,NaN,128.0,0.3,0.001
